In [ ]:
"""
CERN Open Data Portal – Record Schema Inspector
================================================
Importable module for use in Jupyter notebooks.

Quick start:
    from codp_schema_inspector import inspect_schema

    # From a URL
    inspect_schema("https://opendata.cern.ch/record/24130/export/json")

    # From a local file
    inspect_schema("record_24130.json")

    # Show example values alongside types
    inspect_schema("https://opendata.cern.ch/record/24130/export/json", values=True)

    # Sample more array items to catch schema variation
    inspect_schema("https://opendata.cern.ch/record/24130/export/json", sample=3)
"""

import json
import urllib.request
from typing import Any


# ── Type-label helpers ─────────────────────────────────────────────────────────

def _type_label(value: Any) -> str:
    if value is None:            return "null"
    if isinstance(value, bool):  return "bool"
    if isinstance(value, int):   return "int"
    if isinstance(value, float): return "float"
    if isinstance(value, str):   return "str"
    if isinstance(value, list):  return "array"
    if isinstance(value, dict):  return "object"
    return type(value).__name__


def _short_value(value: Any, max_len: int = 60) -> str:
    """Return a short printable representation of a scalar value."""
    if isinstance(value, str):
        s = value if len(value) <= max_len else value[:max_len - 3] + "…"
        return f'"{s}"'
    return str(value)


# ── Schema inference ───────────────────────────────────────────────────────────

def _infer_schema(data: Any, sample: int) -> Any:
    """
    Reduce a JSON value to a lightweight schema descriptor:
      - scalars → their type label string
      - dicts   → {"key": <schema>, ...}
      - arrays  → {"_type": "array", "_length": N, "_item_schema": <merged schema>}
    """
    if isinstance(data, dict):
        return {k: _infer_schema(v, sample) for k, v in data.items()}

    if isinstance(data, list):
        length = len(data)
        if length == 0:
            return {"_type": "array", "_length": 0, "_item_schema": None}
        schemas = [_infer_schema(item, sample) for item in data[:sample]]
        return {"_type": "array", "_length": length, "_item_schema": _merge_schemas(schemas)}

    return _type_label(data)


def _merge_schemas(schemas: list) -> Any:
    """Merge multiple schema descriptors into one representative schema."""
    if not schemas:
        return None
    if all(isinstance(s, str) for s in schemas):
        types = sorted(set(schemas))
        return " | ".join(types)
    if all(isinstance(s, dict) and "_type" not in s for s in schemas):
        all_keys = set().union(*schemas)
        return {k: _merge_schemas([s[k] for s in schemas if k in s])
                for k in sorted(all_keys)}
    return schemas[0]


# ── Pretty-printer ─────────────────────────────────────────────────────────────

def _format_array_header(length: int, item_schema: Any) -> str:
    """Return a compact inline label for an array field."""
    count = f"[{length} item{'s' if length != 1 else ''}]"
    if item_schema is None:
        return f"{count} (empty)"
    if isinstance(item_schema, str):
        return f"{count} of {item_schema}"
    return count  # object/nested array — body printed on next lines


def _print_schema(schema: Any, data: Any, indent: str = "",
                  show_values: bool = False) -> None:
    """Recursively pretty-print a schema tree."""
    if not isinstance(schema, dict):
        # bare scalar type — shouldn't normally be called at top level
        print(f"{indent}{schema}")
        return

    # Array descriptor
    if schema.get("_type") == "array":
        length      = schema["_length"]
        item_schema = schema["_item_schema"]
        header      = _format_array_header(length, item_schema)
        print(header)                        # caller already printed key + ": "
        if isinstance(item_schema, dict):    # object items — recurse
            example_item = data[0] if isinstance(data, list) and data else None
            _print_schema(item_schema, example_item,
                          indent=indent, show_values=show_values)
        return

    # Plain object
    items = list(schema.items())
    for i, (key, sub_schema) in enumerate(items):
        is_last      = i == len(items) - 1
        branch       = "└── " if is_last else "├── "
        child_indent = indent + ("    " if is_last else "│   ")
        child_data   = data.get(key) if isinstance(data, dict) else None

        if isinstance(sub_schema, str):
            # Leaf scalar
            value_hint = ""
            if show_values and child_data is not None and not isinstance(child_data, (dict, list)):
                value_hint = f"  = {_short_value(child_data)}"
            print(f"{indent}{branch}{key}: {sub_schema}{value_hint}")

        elif isinstance(sub_schema, dict) and sub_schema.get("_type") == "array":
            # Inline array header, then optional body on next lines
            length      = sub_schema["_length"]
            item_schema = sub_schema["_item_schema"]
            header      = _format_array_header(length, item_schema)
            print(f"{indent}{branch}{key}: {header}")
            if isinstance(item_schema, dict):   # object items need a body
                example_item = child_data[0] if isinstance(child_data, list) and child_data else None
                _print_schema(item_schema, example_item,
                              indent=child_indent, show_values=show_values)

        else:
            # Nested object — recurse
            print(f"{indent}{branch}{key}:")
            _print_schema(sub_schema, child_data,
                          indent=child_indent, show_values=show_values)


# ── Public API ─────────────────────────────────────────────────────────────────

def inspect_schema(source: str, sample: int = 1, values: bool = False) -> None:
    """
    Print a simplified schema tree for a CERN Open Data Portal record JSON.

    Parameters
    ----------
    source : str
        Either a URL (starting with "http") or a path to a local JSON file.
    sample : int, optional
        Number of array items to sample when inferring the schema (default 1).
        Increase to catch schema variation across array elements.
    values : bool, optional
        If True, print example values next to leaf keys (default False).

    Examples
    --------
    >>> inspect_schema("https://opendata.cern.ch/record/24130/export/json")
    >>> inspect_schema("record_24130.json", sample=3, values=True)
    """
    # ── Load ───────────────────────────────────────────────────────────────────
    if source.startswith("http://") or source.startswith("https://"):
        with urllib.request.urlopen(source) as resp:
            data = json.load(resp)
    else:
        with open(source) as f:
            data = json.load(f)

    # Handle both a single record (dict) and a list of records
    if isinstance(data, list):
        print(f"Array of {len(data)} record(s) – showing schema of first item.\n")
        sample_data = data[0] if data else {}
    else:
        sample_data = data

    schema = _infer_schema(sample_data, sample)

    # ── Header ─────────────────────────────────────────────────────────────────
    record_id = sample_data.get("id", "?")
    title = (sample_data.get("metadata", {}).get("title") or
             sample_data.get("metadata", {}).get("dataset", {}).get("name", ""))
    print(f"CODP Record Schema  (id={record_id})")
    if title:
        print(title)
    print(f"(arrays collapsed to first {sample} item(s))\n")

    _print_schema(schema, sample_data, show_values=values)
    print()

In [ ]:
#from codp_schema_inspector import inspect_schema

# Bare minimum — just a URL
#inspect_schema("https://opendata.cern.ch/record/24130/export/json")
inspect_schema("https://opendata.cern.ch/record/30564/export/json")



In [ ]:
# With example values shown at each leaf
#inspect_schema("https://opendata.cern.ch/record/24130/export/json", values=True)

inspect_schema("https://opendata.cern.ch/record/30564/export/json", values=True)


In [ ]:
# Sample 3 array items to catch any key variation between elements
inspect_schema("https://opendata.cern.ch/record/24130/export/json", sample=3)


In [ ]:

# Works on local files too
inspect_schema("record_24130.json", sample=3, values=True)